[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tranthithuvan555-wq/TEP/blob/main/Einstein_Summation_Notes.ipynb)

# Ghi chú về quy tắc tổng Einstein (Einstein Summation Convention)

Notebook này trình bày:
1. Lịch sử và ý nghĩa của **Quy tắc tổng Einstein**
2. Các phép toán cơ bản với hàm `einsum` trong NumPy/PyTorch
3. Ứng dụng `einsum` trong **Deep Learning** (đặc biệt là Transformer)
4. So sánh `einsum` với các phép toán thông thường

## 0. Cài đặt thư viện cần thiết

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install numpy torch

In [ ]:
# Import thư viện
import numpy as np
import torch
import time

# Đặt seed để tái lập kết quả
np.random.seed(42)
torch.manual_seed(42)

print('NumPy version:', np.__version__)
print('PyTorch version:', torch.__version__)

---
## 1. Giới thiệu Quy tắc tổng Einstein

### 1.1 Lịch sử

**Quy tắc tổng Einstein** (Einstein Summation Convention) được **Albert Einstein** đề xuất năm 1916 trong bài báo về Thuyết Tương đối Rộng. Đây là một quy ước ký hiệu giúp viết gọn các công thức toán học liên quan đến tensor.

**Quy tắc cơ bản:** Nếu một chỉ số xuất hiện **hai lần** trong một biểu thức, ta ngầm hiểu là đang cộng (lấy tổng) trên tất cả giá trị của chỉ số đó.

### 1.2 Quy ước ký hiệu trong `einsum`

Trong NumPy và PyTorch, hàm `einsum` sử dụng **chuỗi ký tự** để mô tả phép toán:

```
np.einsum('ij,jk->ik', A, B)
          └─┬─┘ └─┬─┘ └┬┘
            │     │    └── Kích thước đầu ra
            │     └─────── Kích thước tensor thứ 2
            └───────────── Kích thước tensor thứ 1
```

**Các chỉ số:**
- Chỉ số **xuất hiện ở đầu vào nhưng không ở đầu ra** → **Tổng hóa** (summation)
- Chỉ số **xuất hiện ở cả đầu vào và đầu ra** → **Giữ lại**

Ví dụ: `'ij,jk->ik'` nghĩa là:
- `j` xuất hiện ở đầu vào (`ij`, `jk`) nhưng không ở đầu ra (`ik`) → **Tổng hóa theo j**
- Đây chính là phép **nhân ma trận**: $C_{ik} = \sum_j A_{ij} B_{jk}$

---
## 2. Các phép toán cơ bản với `einsum`

### 2.1 Tích vô hướng (Dot Product)

In [ ]:
# ============================================================
# Tích vô hướng (Dot Product) của hai vector
# Công thức: result = sum(a[i] * b[i])
# Ký hiệu einsum: 'i,i->'
#   - 'i' xuất hiện ở cả hai đầu vào nhưng không ở đầu ra
#   - → tổng hóa theo i → kết quả là một số vô hướng
# ============================================================

a = np.array([1.0, 2.0, 3.0])  # Vector a = [1, 2, 3]
b = np.array([4.0, 5.0, 6.0])  # Vector b = [4, 5, 6]

# Cách 1: Dùng einsum
dot_einsum = np.einsum('i,i->', a, b)

# Cách 2: Dùng np.dot (để so sánh)
dot_standard = np.dot(a, b)

print('=== Tích vô hướng ===')
print(f'a = {a}')
print(f'b = {b}')
print(f'einsum("i,i->", a, b) = {dot_einsum}')
print(f'np.dot(a, b)          = {dot_standard}')
print(f'Tính tay: 1×4 + 2×5 + 3×6 = {1*4 + 2*5 + 3*6}')
print(f'Kết quả khớp nhau: {np.isclose(dot_einsum, dot_standard)}')

### 2.2 Nhân ma trận (Matrix Multiplication)

In [ ]:
# ============================================================
# Nhân ma trận A (m×n) với ma trận B (n×p) → C (m×p)
# Công thức: C[i,k] = sum_j A[i,j] * B[j,k]
# Ký hiệu einsum: 'ij,jk->ik'
#   - j: xuất hiện ở cả hai đầu vào nhưng không ở đầu ra → tổng hóa
#   - i, k: giữ lại ở đầu ra
# ============================================================

A = np.array([[1, 2, 3],
              [4, 5, 6]])  # Ma trận 2×3

B = np.array([[7,  8],
              [9,  10],
              [11, 12]])  # Ma trận 3×2

# Cách 1: Dùng einsum
C_einsum = np.einsum('ij,jk->ik', A, B)

# Cách 2: Dùng np.matmul (để so sánh)
C_standard = np.matmul(A, B)

print('=== Nhân ma trận ===')
print(f'A (kích thước {A.shape}):\n{A}')
print(f'\nB (kích thước {B.shape}):\n{B}')
print(f'\neinsum("ij,jk->ik", A, B) (kích thước {C_einsum.shape}):\n{C_einsum}')
print(f'\nnp.matmul(A, B):\n{C_standard}')
print(f'\nKết quả khớp nhau: {np.allclose(C_einsum, C_standard)}')

### 2.3 Chuyển vị ma trận (Transpose)

In [ ]:
# ============================================================
# Chuyển vị ma trận: A^T[j,i] = A[i,j]
# Ký hiệu einsum: 'ij->ji'
#   - Đổi thứ tự chỉ số: i→j, j→i
#   - Không có tổng hóa
# ============================================================

A = np.array([[1, 2, 3],
              [4, 5, 6]])  # Ma trận 2×3

# Cách 1: Dùng einsum
A_T_einsum = np.einsum('ij->ji', A)

# Cách 2: Dùng .T (để so sánh)
A_T_standard = A.T

print('=== Chuyển vị ma trận ===')
print(f'A (kích thước {A.shape}):\n{A}')
print(f'\neinsum("ij->ji", A) (kích thước {A_T_einsum.shape}):\n{A_T_einsum}')
print(f'\nA.T:\n{A_T_standard}')
print(f'\nKết quả khớp nhau: {np.allclose(A_T_einsum, A_T_standard)}')

### 2.4 Vết ma trận (Trace)

In [ ]:
# ============================================================
# Vết ma trận (Trace): tổng các phần tử trên đường chéo chính
# Công thức: trace(A) = sum_i A[i,i]
# Ký hiệu einsum: 'ii->'
#   - i: xuất hiện hai lần ở đầu vào (→ phần tử đường chéo)
#   - Không có chỉ số ở đầu ra → kết quả là số vô hướng
# ============================================================

A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])  # Ma trận vuông 3×3

# Cách 1: Dùng einsum
trace_einsum = np.einsum('ii->', A)

# Cách 2: Dùng np.trace (để so sánh)
trace_standard = np.trace(A)

print('=== Vết ma trận (Trace) ===')
print(f'A:\n{A}')
print(f'\nĐường chéo chính: {np.diag(A)}')
print(f'\neinsum("ii->", A) = {trace_einsum}')
print(f'np.trace(A)       = {trace_standard}')
print(f'Tính tay: 1 + 5 + 9 = {1 + 5 + 9}')
print(f'Kết quả khớp nhau: {np.isclose(trace_einsum, trace_standard)}')

### 2.5 Tích ngoài (Outer Product)

In [ ]:
# ============================================================
# Tích ngoài (Outer Product) của hai vector
# Công thức: C[i,j] = a[i] * b[j]
# Ký hiệu einsum: 'i,j->ij'
#   - Không có chỉ số bị tổng hóa
#   - i và j đều xuất hiện ở đầu ra
# ============================================================

a = np.array([1.0, 2.0, 3.0])  # Vector 3 chiều
b = np.array([4.0, 5.0])        # Vector 2 chiều

# Cách 1: Dùng einsum
outer_einsum = np.einsum('i,j->ij', a, b)

# Cách 2: Dùng np.outer (để so sánh)
outer_standard = np.outer(a, b)

print('=== Tích ngoài (Outer Product) ===')
print(f'a = {a}')
print(f'b = {b}')
print(f'\neinsum("i,j->ij", a, b) (kích thước {outer_einsum.shape}):')
print(outer_einsum)
print(f'\nnp.outer(a, b):')
print(outer_standard)
print(f'\nKết quả khớp nhau: {np.allclose(outer_einsum, outer_standard)}')

### 2.6 Tổng theo trục (Sum along axis)

In [ ]:
# ============================================================
# Tổng theo trục (Axis Sum)
#   - Tổng tất cả: 'ij->'  (tổng toàn bộ phần tử)
#   - Tổng theo hàng: 'ij->i'  (tổng mỗi hàng)
#   - Tổng theo cột: 'ij->j'  (tổng mỗi cột)
# ============================================================

A = np.array([[1, 2, 3],
              [4, 5, 6]])  # Ma trận 2×3

sum_all   = np.einsum('ij->', A)    # Tổng tất cả → số vô hướng
sum_rows  = np.einsum('ij->i', A)   # Tổng theo cột → vector theo hàng
sum_cols  = np.einsum('ij->j', A)   # Tổng theo hàng → vector theo cột

print('=== Tổng theo trục ===')
print(f'A:\n{A}')
print(f'\neinsum("ij->", A) = {sum_all}  (tổng tất cả phần tử)')
print(f'np.sum(A)         = {A.sum()}')
print()
print(f'einsum("ij->i", A) = {sum_rows}  (tổng từng hàng)')
print(f'A.sum(axis=1)     = {A.sum(axis=1)}')
print()
print(f'einsum("ij->j", A) = {sum_cols}  (tổng từng cột)')
print(f'A.sum(axis=0)     = {A.sum(axis=0)}')

### 2.7 Phần tử theo phần tử (Element-wise) và Hadamard Product

In [ ]:
# ============================================================
# Tích Hadamard: nhân từng phần tử tương ứng
# Công thức: C[i,j] = A[i,j] * B[i,j]
# Ký hiệu einsum: 'ij,ij->ij'
#   - Giữ lại tất cả chỉ số ở đầu ra
#   - Không có tổng hóa
# ============================================================

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# Tích Hadamard
hadamard_einsum = np.einsum('ij,ij->ij', A, B)
hadamard_standard = A * B

# Tích Frobenius (nhân từng phần tử rồi tổng tất cả)
frob_einsum = np.einsum('ij,ij->', A, B)
frob_standard = np.sum(A * B)

print('=== Tích Hadamard và Frobenius ===')
print(f'A:\n{A}')
print(f'B:\n{B}')
print(f'\nTích Hadamard einsum("ij,ij->ij"):\n{hadamard_einsum}')
print(f'A * B:\n{hadamard_standard}')
print(f'\nTích Frobenius einsum("ij,ij->") = {frob_einsum}')
print(f'np.sum(A * B)                   = {frob_standard}')

---
## 3. Ứng dụng `einsum` trong Deep Learning

### 3.1 Tính Attention Scores trong Transformer

Trong Self-Attention, ta cần tính: $\text{scores} = QK^T$

Với batch processing, Q và K có kích thước `(batch, seq_len, d_k)`, ta cần tính tích vô hướng giữa mỗi cặp query-key.

In [ ]:
# ============================================================
# Tính Attention Scores: QK^T
# Q, K có kích thước (batch, seq_len, d_k)
# Kết quả: (batch, seq_len_q, seq_len_k)
# 
# einsum: 'bqd,bkd->bqk'
#   - b: batch dimension (giữ lại)
#   - q: query position (giữ lại)
#   - k: key position (giữ lại)
#   - d: embedding dimension (tổng hóa → dot product)
# ============================================================

import math

# Tạo dữ liệu giả
batch_size = 2
seq_len = 5    # Độ dài chuỗi
d_k = 8        # Số chiều key/query

Q = np.random.randn(batch_size, seq_len, d_k)  # (2, 5, 8)
K = np.random.randn(batch_size, seq_len, d_k)  # (2, 5, 8)

# Cách 1: Dùng einsum
scores_einsum = np.einsum('bqd,bkd->bqk', Q, K) / math.sqrt(d_k)

# Cách 2: Dùng np.matmul (để so sánh)
scores_matmul = np.matmul(Q, K.transpose(0, 2, 1)) / math.sqrt(d_k)

print('=== Attention Scores: QK^T ===')
print(f'Q kích thước: {Q.shape}  → (batch, seq_len_q, d_k)')
print(f'K kích thước: {K.shape}  → (batch, seq_len_k, d_k)')
print(f'\nAttention scores (einsum): {scores_einsum.shape}  → (batch, seq_q, seq_k)')
print(f'Kết quả khớp nhau: {np.allclose(scores_einsum, scores_matmul)}')
print(f'\nMa trận scores (batch=0, sau chia sqrt(d_k)):')
print(np.round(scores_einsum[0], 3))

### 3.2 Multi-Head Attention với einsum

In [ ]:
# ============================================================
# Multi-Head Attention Scores
# Q, K có kích thước (batch, heads, seq_len, d_k)
#
# einsum: 'bhqd,bhkd->bhqk'
#   - b: batch (giữ lại)
#   - h: head (giữ lại)
#   - q: query position (giữ lại)
#   - k: key position (giữ lại)
#   - d: head dimension (tổng hóa)
# ============================================================

batch_size = 2
num_heads = 4
seq_len = 6
d_head = 16   # Số chiều mỗi head

Q_mh = np.random.randn(batch_size, num_heads, seq_len, d_head)
K_mh = np.random.randn(batch_size, num_heads, seq_len, d_head)
V_mh = np.random.randn(batch_size, num_heads, seq_len, d_head)

# Tính attention scores cho tất cả heads cùng lúc
scores_mh = np.einsum('bhqd,bhkd->bhqk', Q_mh, K_mh) / math.sqrt(d_head)

# Softmax để lấy attention weights
scores_exp = np.exp(scores_mh - scores_mh.max(axis=-1, keepdims=True))
attn_weights = scores_exp / scores_exp.sum(axis=-1, keepdims=True)

# Tính context vector: weighted sum of values
# einsum: 'bhqk,bhkd->bhqd'
context = np.einsum('bhqk,bhkd->bhqd', attn_weights, V_mh)

print('=== Multi-Head Attention với einsum ===')
print(f'Q/K/V kích thước: {Q_mh.shape}  → (batch, heads, seq, d_head)')
print(f'Attention scores: {scores_mh.shape}')
print(f'Attention weights: {attn_weights.shape}')
print(f'Context vector:    {context.shape}')
print(f'\nKiểm tra: Mỗi hàng attention weights có tổng = 1?')
print(f'  Tổng weights [batch=0, head=0, query=0]: {attn_weights[0, 0, 0].sum():.6f}')

### 3.3 Batch Matrix Multiplication

In [ ]:
# ============================================================
# Batch Matrix Multiplication (BMM)
# Nhân nhiều cặp ma trận cùng lúc (theo batch)
#
# A: (batch, m, n)  ×  B: (batch, n, p)  →  C: (batch, m, p)
# einsum: 'bmn,bnp->bmp'
# ============================================================

batch = 3
m, n, p = 4, 5, 6

A_batch = np.random.randn(batch, m, n)  # (3, 4, 5)
B_batch = np.random.randn(batch, n, p)  # (3, 5, 6)

# Cách 1: einsum
C_einsum = np.einsum('bmn,bnp->bmp', A_batch, B_batch)

# Cách 2: np.matmul (để so sánh)
C_matmul = np.matmul(A_batch, B_batch)

print('=== Batch Matrix Multiplication ===')
print(f'A: {A_batch.shape}  → (batch, m, n)')
print(f'B: {B_batch.shape}  → (batch, n, p)')
print(f'C: {C_einsum.shape}  → (batch, m, p)')
print(f'\nKết quả khớp nhau: {np.allclose(C_einsum, C_matmul)}')
print(f'\neinsum xử lý {batch} cặp ma trận ({m}×{n}) × ({n}×{p}) cùng lúc!')

### 3.4 Tensor Contraction (Co-rút tensor)

In [ ]:
# ============================================================
# Tensor Contraction: Tổng hóa nhiều chiều cùng lúc
# Ví dụ: Bilinear product trong mạng nơ-ron
#
# Tính: output[b] = x[b,i] * W[i,j] * y[b,j]
# einsum: 'bi,ij,bj->b'
# ============================================================

batch = 4
d_in = 8

x = np.random.randn(batch, d_in)    # (4, 8)
W = np.random.randn(d_in, d_in)     # (8, 8)
y = np.random.randn(batch, d_in)    # (4, 8)

# Bilinear product: x^T W y cho từng mẫu trong batch
bilinear = np.einsum('bi,ij,bj->b', x, W, y)

# Cách không dùng einsum (để so sánh)
bilinear_manual = np.array([x[b] @ W @ y[b] for b in range(batch)])

print('=== Tensor Contraction: Bilinear Product ===')
print(f'x: {x.shape}  (inputs)')
print(f'W: {W.shape}  (weight matrix)')
print(f'y: {y.shape}  (outputs)')
print(f'\nBilinear product: {bilinear}')
print(f'Kết quả khớp nhau: {np.allclose(bilinear, bilinear_manual)}')
print(f'\neinsum xử lý toàn bộ batch ({batch} mẫu) chỉ trong 1 dòng!')

---
## 4. So sánh `einsum` với các phép toán thông thường

### 4.1 Hiệu suất

In [ ]:
# ============================================================
# So sánh tốc độ: einsum vs matmul cho nhân ma trận lớn
# ============================================================

import time

# Tạo ma trận lớn
n = 500
A_large = np.random.randn(n, n)
B_large = np.random.randn(n, n)

num_trials = 10

# Đo thời gian einsum
start = time.time()
for _ in range(num_trials):
    _ = np.einsum('ij,jk->ik', A_large, B_large)
time_einsum = (time.time() - start) / num_trials * 1000  # ms

# Đo thời gian matmul
start = time.time()
for _ in range(num_trials):
    _ = np.matmul(A_large, B_large)
time_matmul = (time.time() - start) / num_trials * 1000  # ms

print('=== So sánh tốc độ: Nhân ma trận 500×500 ===')
print(f'einsum:  {time_einsum:.2f} ms')
print(f'matmul:  {time_matmul:.2f} ms')
print()
if time_matmul < time_einsum:
    print(f'→ matmul nhanh hơn {time_einsum/time_matmul:.1f}x')
    print('→ np.matmul dùng BLAS/LAPACK được tối ưu hóa cao')
else:
    print(f'→ einsum nhanh hơn {time_matmul/time_einsum:.1f}x')
print()
print('Lưu ý: Với phép nhân ma trận đơn giản, np.matmul/torch.matmul')
print('thường nhanh hơn einsum. Dùng einsum khi cần viết phép toán phức tạp.')

### 4.2 So sánh với PyTorch `torch.einsum`

In [ ]:
# ============================================================
# torch.einsum có cú pháp giống NumPy và hỗ trợ autograd
# (quan trọng cho deep learning)
# ============================================================

# Tạo tensor PyTorch với gradient tracking
Q_torch = torch.randn(2, 4, 8, requires_grad=True)  # (batch, seq, d)
K_torch = torch.randn(2, 4, 8, requires_grad=True)  # (batch, seq, d)

# Tính attention scores với torch.einsum
scores_torch = torch.einsum('bqd,bkd->bqk', Q_torch, K_torch)

# Tính loss giả và backprop
loss = scores_torch.mean()
loss.backward()  # Tự động tính gradient!

print('=== torch.einsum với Autograd ===')
print(f'Q: {Q_torch.shape}')
print(f'K: {K_torch.shape}')
print(f'Attention scores: {scores_torch.shape}')
print(f'\nGradient tự động được tính:')
print(f'dL/dQ gradient shape: {Q_torch.grad.shape}')
print(f'dL/dK gradient shape: {K_torch.grad.shape}')
print(f'\nNhận xét: torch.einsum tự động tính gradient qua backpropagation!')

### 4.3 Bảng tóm tắt các phép toán einsum

In [ ]:
# ============================================================
# Bảng tóm tắt: Cú pháp einsum cho các phép toán phổ biến
# ============================================================

summary = [
    ('Tích vô hướng (Dot product)',   'i,i->',      '(n,), (n,) → scalar'),
    ('Tích ngoài (Outer product)',    'i,j->ij',    '(m,), (n,) → (m,n)'),
    ('Nhân ma trận (MatMul)',          'ij,jk->ik',  '(m,n), (n,p) → (m,p)'),
    ('Chuyển vị (Transpose)',          'ij->ji',     '(m,n) → (n,m)'),
    ('Vết ma trận (Trace)',            'ii->',       '(n,n) → scalar'),
    ('Tổng tất cả',                   'ij->',       '(m,n) → scalar'),
    ('Tổng theo hàng',                'ij->i',      '(m,n) → (m,)'),
    ('Tổng theo cột',                 'ij->j',      '(m,n) → (n,)'),
    ('Tích Hadamard',                 'ij,ij->ij',  '(m,n), (m,n) → (m,n)'),
    ('Batch MatMul',                  'bmn,bnp->bmp','(b,m,n), (b,n,p) → (b,m,p)'),
    ('Attention scores (Q·K^T)',      'bqd,bkd->bqk','(b,q,d), (b,k,d) → (b,q,k)'),
    ('MH Attention scores',           'bhqd,bhkd->bhqk','(b,h,q,d), (b,h,k,d) → (b,h,q,k)'),
    ('Context vector (attn·V)',       'bhqk,bhkd->bhqd','(b,h,q,k), (b,h,k,d) → (b,h,q,d)'),
]

print('=== Bảng tóm tắt: Cú pháp einsum phổ biến ===')
print(f'{"Phép toán":<40} {"Cú pháp einsum":<30} {"Kích thước"}')
print('-' * 100)
for name, syntax, shapes in summary:
    print(f'{name:<40} {syntax:<30} {shapes}')

---
## 5. Khi nào nên dùng `einsum`?

### Nên dùng `einsum` khi:
- Phép toán phức tạp với nhiều chiều (tensor >2D)
- Muốn code **dễ đọc** và **dễ hiểu** (biểu thức toán học trực quan)
- Kết hợp nhiều phép toán trong một lần (tích + tổng hóa)
- Prototype hoặc giảng dạy/nghiên cứu

### Có thể dùng phương án khác khi:
- Phép nhân ma trận đơn giản → `torch.matmul` hoặc `@` nhanh hơn
- Tổng đơn giản → `torch.sum(x, dim=...)` rõ ràng hơn
- Cần hiệu suất tối đa → kiểm tra cả hai và chọn cái nhanh hơn

In [ ]:
# ============================================================
# Ví dụ cuối: Tổng hợp - Tính toàn bộ Scaled Dot-Product Attention
# bằng einsum trong vài dòng code
# ============================================================

def scaled_dot_product_attention_einsum(Q, K, V, mask=None):
    """
    Tính Scaled Dot-Product Attention hoàn toàn bằng einsum.
    
    Q, K, V: numpy arrays, kích thước (batch, seq_len, d_k)
    """
    d_k = Q.shape[-1]
    
    # Bước 1: Tính attention scores: QK^T / sqrt(d_k)
    scores = np.einsum('bqd,bkd->bqk', Q, K) / math.sqrt(d_k)
    
    # Bước 2: Áp dụng mask (nếu có)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    
    # Bước 3: Softmax
    scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
    attn_weights = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
    
    # Bước 4: Tổng hóa với Value
    output = np.einsum('bqk,bkd->bqd', attn_weights, V)
    
    return output, attn_weights


# Kiểm tra
batch, seq_len, d_k = 2, 6, 16
Q = np.random.randn(batch, seq_len, d_k)
K = np.random.randn(batch, seq_len, d_k)
V = np.random.randn(batch, seq_len, d_k)

output, weights = scaled_dot_product_attention_einsum(Q, K, V)

print('=== Scaled Dot-Product Attention bằng einsum ===')
print(f'Q/K/V: {Q.shape}')
print(f'Output: {output.shape}')
print(f'Attention weights: {weights.shape}')
print(f'Tổng weights mỗi query = 1: {np.allclose(weights.sum(axis=-1), 1.0)}')
print()
print('Chỉ với 4 dòng einsum, ta có toàn bộ cơ chế Attention!')

---
## 6. Tổng kết

### Những gì đã học:

| Phép toán | Cú pháp einsum | Ghi chú |
|---|---|---|
| Dot product | `'i,i->'` | Tổng hóa chỉ số i |
| Outer product | `'i,j->ij'` | Không có tổng hóa |
| Matrix multiply | `'ij,jk->ik'` | Tổng hóa j |
| Transpose | `'ij->ji'` | Hoán vị chỉ số |
| Batch matmul | `'bmn,bnp->bmp'` | Giữ batch dim b |
| Attention scores | `'bqd,bkd->bqk'` | Tổng hóa d (embedding dim) |

### Quy tắc nhớ nhanh:
1. **Chỉ số lặp lại** trong đầu vào nhưng **không có ở đầu ra** → **Tổng hóa**
2. **Chỉ số chỉ xuất hiện một lần** ở đầu vào và **có ở đầu ra** → **Giữ lại**
3. **Thứ tự chỉ số ở đầu ra** quyết định **hình dạng của kết quả**

### Tài liệu tham khảo:
- [NumPy einsum documentation](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html)
- [PyTorch einsum documentation](https://pytorch.org/docs/stable/generated/torch.einsum.html)
- [Einsum is All you Need – Medium article](https://rockt.github.io/2018/04/30/einsum)
- [Einstein Summation Convention – Wikipedia](https://en.wikipedia.org/wiki/Einstein_notation)